# 02 · Thinking in N dimensions / Pensar en N dimensiones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2">PART II · DEMO + EXERCISE · 20 MIN</span>

## Practise today / Practica hoy

Keep images and labels paired when shuffling; explain why shuffling time changes a sequence.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Mantener imágenes y etiquetas emparejadas al barajar; explicar por qué barajar el tiempo cambia una secuencia.</div></div>

## Explore later / Explora después

Build padded video batches and interpret additional experimental axes.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Construir lotes de video con relleno e interpretar otros ejes experimentales.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 01: can two order-3 arrays assign different meanings to axis 0? Give an example.

🇪🇸 Recuerda el cuaderno 01: ¿dos arrays de orden 3 pueden dar significados distintos al eje 0? Da un ejemplo.

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Three real datasets: handwritten digits, one RGB photograph, and a CC0 storm
video.

The video is downloaded once and checked with SHA-256, so we know exactly which
file we have.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tres conjuntos reales: dígitos manuscritos, una fotografía RGB y un video de tormenta CC0. El video se descarga una vez y se verifica con SHA-256, así sabemos exactamente qué archivo tenemos.</div>

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

# Enable widgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

rng = np.random.default_rng(0)

# ---------------------------------------------------------------------------
# Real image data
# ---------------------------------------------------------------------------
digits = load_digits()
digit_batch = digits.images[:8].astype(np.float32)   # (N, H, W)
digit_labels = digits.target[:8]
real_digit = digit_batch[0]
real_photo = data.astronaut()                        # (H, W, C)

# ---------------------------------------------------------------------------
# Real video data: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0
# ---------------------------------------------------------------------------
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
VIDEO_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    frames = []
    source_indices = []

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        if i % stride == 0:
            frames.append(frame)
            source_indices.append(i)
            if len(frames) == n_frames:
                break

    return np.stack(frames), np.asarray(source_indices)

real_video, source_indices = fetch_verified_video(VIDEO_URL, VIDEO_SHA256)

assert real_video.shape == (16, 540, 960, 3), real_video.shape

# Same numerical shape as digit_batch: (8, 8, 8), but different semantics.
r0 = real_video.shape[1] // 2 - 4
c0 = real_video.shape[2] // 2 - 4
video_patch = (
    real_video[:8, r0:r0 + 8, c0:c0 + 8]
    .mean(axis=3)
    .astype(np.float32)
)

print("EN: Setup ready with real image and video data.")
print("ES: Preparación lista con datos reales de imágenes y video.")
print()
print("real_digit / dígito real:", real_digit.shape)
print("digit_batch / lote de dígitos:", digit_batch.shape)
print("real_photo / foto real:", real_photo.shape)
print("real_video / video real:", real_video.shape)
print("video_patch / recorte temporal:", video_patch.shape)

## The five axis letters / Las cinco letras de ejes

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

| Letter / Letra | English | Español | The question it answers / La pregunta que responde |
|---|---|---|---|
| `N` | batch / examples | lote / ejemplos | How many independent examples? / ¿Cuántos ejemplos independientes? |
| `T` | time | tiempo | How many measured moments? / ¿Cuántos momentos medidos? |
| `H` | height | alto | How many pixel rows? / ¿Cuántas filas de píxeles? |
| `W` | width | ancho | How many pixel columns? / ¿Cuántas columnas de píxeles? |
| `C` | channels | canales | How many colour or measurement channels? / ¿Cuántos canales? |

Read <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N, T, H, W, C)</span> as a sentence and it stops being cryptic:
**examples × time × height × width × channels**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Lee <code>(N, T, H, W, C)</code> como una frase — <b>ejemplos × tiempo × alto × ancho × canales</b> — y deja de ser críptica.</div>

## 2.2 Same shape, different meaning / Misma forma, distinto significado

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

The most important comparison in this notebook:

<div style="margin:1.4em 0;font:400 15px/2.4 ui-sans-serif,system-ui,sans-serif">
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">digit_batch · (8, 8, 8)</span> &nbsp;→&nbsp; <b>(N, H, W)</b><br>
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">video_patch · (8, 8, 8)</span> &nbsp;→&nbsp; <b>(T, H, W)</b>
</div>

The first axis has the same **size**. It does not have the same **role**.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">EIGHT CARDS · OCHO TARJETAS</div><div style="margin:.55em 0"><b>Batch</b> — eight separate postcards. Reorder them and you still have the same set.</div><div style="margin:.55em 0"><b>Time</b> — eight frames of an animation. Reorder them and the motion is wrong.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El primer eje tiene el mismo <b>tamaño</b>, no el mismo <b>papel</b>. <b>Lote:</b> ocho postales sueltas; cambiar el orden no cambia el conjunto. <b>Tiempo:</b> ocho fotogramas; cambiar el orden estropea el movimiento.</div>

### Shuffle it, and watch which reading breaks / Reorganízalo y mira qué lectura se rompe

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Two readings, one permutation. The four planes are reordered once — nothing added, nothing removed — and then each reading is put against it. The average cannot tell that anything happened. The series through a single cell can tell immediately. That is the whole difference between a batch axis and a time axis.

$$
\frac{1}{N}\sum_{b=0}^{N-1} x_{\pi(b)} \;=\; \frac{1}{N}\sum_{b=0}^{N-1} x_{b}
\qquad\text{and yet}\qquad
x_{\pi(t+1)} - x_{\pi(t)} \;\neq\; x_{t+1} - x_{t}
$$

Read it as: a sum does not care what order it adds things in, so every
per-example statistic survives a shuffle untouched — that is the left-hand
side, and it is the third frame. A difference between neighbours is *made* of
the order, so it does not survive at all. The shape cannot tell you which of
the two axes you are holding. You can.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-02-shuffle.gif" alt="An animation of four matrices in a row, each shaded one step darker than the last. They are reordered by a permutation and the shading is visibly out of sequence. The mean over the four is then shown to be identical before and after, while the four values read from one cell run 0, 10, 20, 30 before and 20, 0, 30, 10 after." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>Dos lecturas, una sola permutación. Los cuatro planos se reordenan una vez —sin añadir ni quitar nada— y después se enfrenta cada lectura con ella. El promedio no puede notar que haya pasado nada. La serie que atraviesa una sola celda lo nota de inmediato. Esa es toda la diferencia entre un eje de lote y un eje de tiempo.</div>

## Exercise 2 — shuffle batch vs. shuffle time / Ejercicio 2 — reorganiza lote vs. tiempo

Apply the **same permutation** to the digit batch and video sequence; move digit labels with their images.
Record what you expect to change before running. Then compare the **mean absolute change between consecutive sampled frames** and the retained source indices. A change metric alone cannot establish chronology.

🇪🇸 Aplica la **misma permutación** al lote de dígitos y a la secuencia de video; mueve las etiquetas con sus imágenes.
Anota qué esperas que cambie antes de ejecutar. Después compara el **cambio absoluto medio entre fotogramas muestreados consecutivos** y los índices originales conservados. Una métrica de cambio por sí sola no establece la cronología.


### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

Track image-label pairs as units. Separately, track which original frame occupies each new time position.

🇪🇸 Sigue los pares imagen-etiqueta como unidades. Por separado, sigue qué fotograma original ocupa cada nueva posición temporal.

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

Index both arrays with the same perm. For consecutive changes use np.abs(frames[1:] - frames[:-1]).mean(); test recovery with np.argsort(perm).

🇪🇸 Indexa ambos arrays con la misma perm. Para cambios consecutivos usa np.abs(frames[1:] - frames[:-1]).mean(); comprueba la recuperación con np.argsort(perm).

</details>



### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** What changes if you shuffle images? What changes if you shuffle frames?
2. **Run.** Complete Exercise 2.
3. **Explain.** Explain why matching shapes do not imply matching meanings.
4. **Check.** Undo perm with np.argsort(perm). Confirm the digit-label pairs return. Do not treat a change metric as proof of chronology.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿Qué cambia al barajar imágenes? ¿Y fotogramas?
2. **Ejecuta.** Completa el Ejercicio 2.
3. **Explica.** Explica por qué formas iguales no implican significados iguales.
4. **Comprueba.** Deshaz perm con np.argsort(perm). Comprueba los pares imagen-etiqueta. Una métrica de cambio no prueba la cronología.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Create perm = rng.permutation(8).
# 2. Apply it to digit_batch AND digit_labels.
# 3. Apply it to video_patch.
# 4. Print original and shuffled labels.
# 5. Compare the mean absolute change between consecutive sampled video frames.
# 6. Explain why the digit set is still the same but the video chronology is not.
#
# ES:
# 1. Crea perm = rng.permutation(8).
# 2. Aplícala a digit_batch Y digit_labels.
# 3. Aplícala a video_patch.
# 4. Imprime etiquetas originales y reorganizadas.
# 5. Compara el cambio absoluto medio entre fotogramas muestreados consecutivos.
# 6. Explica por qué el conjunto de dígitos sigue siendo el mismo pero la cronología del video no.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

perm = rng.permutation(8)

shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

inverse_perm = np.argsort(perm)
same_examples = (
    np.array_equal(shuffled_digits[inverse_perm], digit_batch)
    and np.array_equal(shuffled_labels[inverse_perm], digit_labels)
)
assert same_examples


def mean_consecutive_sampled_change(x):
    x = x.astype(np.float32)
    return float(np.mean(np.abs(x[1:] - x[:-1])))

before = mean_consecutive_sampled_change(video_patch)
after = mean_consecutive_sampled_change(shuffled_video)

print("Permutation / Permutación:", perm.tolist())
print("Original labels / Etiquetas originales:", digit_labels.tolist())
print("Shuffled labels / Etiquetas reorganizadas:", shuffled_labels.tolist())
print("Same labeled examples / Mismos ejemplos etiquetados:", same_examples)
print()

print(f"Video change before / Cambio antes: {before:.3f}")
print(f"Video change after  / Cambio después: {after:.3f}")
print(f"After/before ratio / Razón después/antes: {after / before:.2f}x")
print()

print("EN: the digit examples are the same; only their presentation order changed.")
print("ES: los ejemplos de dígitos son los mismos; solo cambió su orden de presentación.")
print("Original frame indices / Índices originales:", source_indices[:8].tolist())
print("Shuffled frame indices / Índices reorganizados:", source_indices[:8][perm].tolist())
print("EN: use source indices to inspect chronology; mean change alone cannot prove it.")
print("ES: usa los índices originales para revisar la cronología; el cambio medio por sí solo no la prueba.")

<details>
<summary><strong>What did the shuffle prove? / ¿Qué demostró la reorganización?</strong></summary>

For the digits, the order of independent examples changed — and every image
kept its label.

For the video, the same measured frames stayed, and the chronology did not.

**An operation can leave `shape` and `dtype` untouched and still change the
scientific meaning.**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>En los dígitos cambió el orden de ejemplos independientes y cada imagen conservó su etiqueta. En el video quedaron los mismos fotogramas y se alteró la cronología. <b>Una operación puede dejar intactos <code>shape</code> y <code>dtype</code> y aun así cambiar el significado científico.</b></div>

</details>

### Group discussion / Discusión en grupo

**Time:** 6 minutes.

Two arrays have shape `(20, 8, 8)`. One contains independent digit images.
The other contains consecutive video frames. Propose an operation that is
reasonable for one task but misleading for the other.

- What happens if you shuffle or average the first axis?
- What question would make averaging useful? What question would it destroy?
- What metadata must travel with the array?

**Share:** Two interpretations of the same operation. Include a condition under
which your recommendation would change.

<details>
<summary>Español</summary>

**Tiempo:** 6 minutos.

Dos arrays tienen forma `(20, 8, 8)`. Uno contiene imágenes independientes de
dígitos; el otro, fotogramas consecutivos. Propongan una operación razonable
para una tarea pero engañosa para la otra.

- ¿Qué ocurre al mezclar o promediar el primer eje?
- ¿Para qué pregunta serviría promediar? ¿Qué pregunta impediría responder?
- ¿Qué metadatos deben acompañar al array?

**Compartan:** Dos interpretaciones de la misma operación. Incluyan una
condición que cambiaría su recomendación.

</details>

### Checkpoint / Comprobación

Record your group’s two interpretations of shuffling or averaging axis 0. Which metadata would change your recommendation?

🇪🇸 Anota las dos interpretaciones grupales de barajar o promediar el eje 0. ¿Qué metadatos cambiarían tu recomendación?

Answer / Respuesta: ___

<details>
<summary>Check after attempting / Comprueba después de intentarlo</summary>

A permutation can preserve a labeled image set when labels move with it, while changing temporal adjacency in a sequence. State the task before judging an average.

🇪🇸 Una permutación puede conservar un conjunto de imágenes etiquetadas si las etiquetas se mueven con él, pero cambia la adyacencia temporal de una secuencia. Define la tarea antes de juzgar un promedio.

</details>

## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 03 / Siguiente: cuaderno 03](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;The average is identical after shuffling, so <b>no information was lost</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como la <b>media</b> no cambia al reorganizar, no se perdió información. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: does shuffling lose anything? / Predice: ¿se pierde algo al reorganizar? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

pred_frames = np.array([0., 1., 2., 3.])
pred_shuffled = pred_frames[[0, 3, 1, 2]]

assert pred_frames.mean() == pred_shuffled.mean()
assert not np.array_equal(
    np.diff(pred_frames), np.diff(pred_shuffled)
)
# --- end counterexample / fin del contraejemplo ---

# Imported here rather than leaned on from Setup: every other
# predict cell imports what it uses, and this one is early enough
# that a reader can reach it before running anything above.
import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#0891b2"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Nothing is lost — the mean proves it / No se pierde nada — la media lo prueba", "nothing"),
        ("Both the mean and the order change / Cambian la media y el orden", "both"),
        ("The mean survives, the order does not / La media sobrevive, el orden no", "mean_only"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Original / Original:", pred_frames)
    print("Shuffled / Reorganizado:", pred_shuffled)
    print()
    print("Mean / Media:", pred_frames.mean(), "->", pred_shuffled.mean())
    print("Steps / Pasos:", np.diff(pred_frames), "->", np.diff(pred_shuffled))
    print()
    if choice == "mean_only":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: the mean is unchanged, but the step pattern is not. A mean cannot establish chronology — check timestamps or source indices.")
    print("ES: la media no cambia, pero el patrón de pasos sí. Una media no puede establecer la cronología: revisa marcas de tiempo o índices de origen.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## Shape is not semantics / La forma no es la semántica

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

`digit_batch` and `video_patch` are both <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(8, 8, 8)</span>. One is eight
independent images. The other is eight ordered moments.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">THE WHOLE NOTEBOOK IN ONE LINE · TODO EL CUADERNO EN UNA LÍNEA</div>An operation can be <b>legal</b> for both shapes; whether it is useful depends on what each axis means.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>digit_batch</code> y <code>video_patch</code> son ambos <code>(8, 8, 8)</code>: uno son ocho imágenes independientes y el otro ocho instantes ordenados. Una operación puede ser <b>legal</b> para ambas formas; su utilidad depende del significado de cada eje.</div>

### Stack against concatenate / Stack frente a concatenate

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Both take the same three matrices and both return sixty numbers. `np.stack` adds a new axis and the frames stay separable; `np.concatenate` joins along an axis that already exists, and the boundaries between frames are gone.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-02-stack.gif" alt="An animation showing three 4 by 5 matrices apart, then stacked into a 3 by 4 by 5 pile, then concatenated into a single 12 by 5 matrix, then the two results side by side." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ambas toman las mismas tres matrices y ambas devuelven sesenta números. <code>np.stack</code> añade un eje nuevo y los fotogramas siguen separables; <code>np.concatenate</code> une a lo largo de un eje que ya existe, y los límites entre fotogramas desaparecen.</div>

### Padding, and the zeros nobody measured / Padding y los ceros que nadie midió

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Three clips, three lengths, one batch. Every clip is stretched to the longest, and the planes that were never recorded are drawn as dashed outlines rather than filled, because that is what they are. The last frame is why a mask is not optional: averaging the first clip over four slots instead of two halves every number in it, exactly.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-02-pad.gif" alt="An animation of three video clips of two, four and three frames. Each is padded out to four frames, with the invented frames drawn as dashed empty outlines. A three by four mask then shows a one wherever a frame was measured and a zero where it was not. The last frame shows the mean of the first clip over its two real frames, three, beside its mean over all four padded slots, one point five." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>Tres clips, tres longitudes, un solo lote. Cada clip se estira hasta el más largo, y los planos que nunca se grabaron se dibujan como contornos discontinuos en vez de rellenos, porque eso es lo que son. El último fotograma explica por qué la máscara no es opcional: promediar el primer clip sobre cuatro posiciones en lugar de dos reduce a la mitad cada número.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop forever and a GIF cannot
# be paused — so this fetches the same frames and hands them over one at a
# time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-02-shuffle.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-02-stack.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-02-pad.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


## 2.1 Read real tensors as sentences / Lee tensores reales como frases

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Real objects, not empty arrays.

| Real object / Objeto real | Shape / Forma | Order / Orden | Read it as / Léelo como |
|---|---:|---:|---|
| one digit / un dígito | `(8, 8)` | 2 | height × width / alto × ancho |
| digit batch / lote de dígitos | `(8, 8, 8)` | 3 | examples × height × width / ejemplos × alto × ancho |
| RGB photo / foto RGB | `(512, 512, 3)` | 3 | height × width × colour / alto × ancho × color |
| sampled video / video muestreado | `(16, 540, 960, 3)` | 4 | time × height × width × colour / tiempo × alto × ancho × color |
| padded video batch / lote con padding | `(N, T, H, W, C)` | 5 | examples × time × height × width × colour |

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">NOT A LADDER OF QUALITY · NO ES UNA ESCALA DE CALIDAD</div>A higher order is not better, or cleverer. It is <b>more axes</b>. Nothing else.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un orden mayor no significa «mejor» ni «más inteligente». Significa <b>más ejes</b>, y nada más.</div>

### Axis explorer / Explorador de ejes

Pick a real object. The notebook turns its shape into a sentence.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un objeto real: el cuaderno traduce su forma a una frase.</div>

In [ ]:
#@title 🔍 Axis explorer / Explorador de ejes — run me / ejecútame { display-mode: 'form' }

axis_choice = widgets.Dropdown(
    options=[
        ("One digit / Un dígito", "digit"),
        ("Digit batch / Lote de dígitos", "batch"),
        ("RGB photo / Foto RGB", "photo"),
        ("Real video / Video real", "video"),
    ],
    value="batch",
    description="Object / Objeto:",
    style={"description_width": "120px"},
)

def explain_real_tensor(choice):
    items = {
        "digit": (
            real_digit,
            "(H, W)",
            "height × width",
            "alto × ancho",
        ),
        "batch": (
            digit_batch,
            "(N, H, W)",
            "examples × height × width",
            "ejemplos × alto × ancho",
        ),
        "photo": (
            real_photo,
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "video": (
            real_video,
            "(T, H, W, C)",
            "time × height × width × colour",
            "tiempo × alto × ancho × color",
        ),
    }

    arr, symbols, en, es = items[choice]

    print("Shape / Forma:", arr.shape)
    print("Order / Orden:", arr.ndim)
    print("Axes / Ejes:", symbols)
    print("EN:", en)
    print("ES:", es)

axis_output = widgets.interactive_output(
    explain_real_tensor,
    {"choice": axis_choice},
)

display(widgets.VBox([axis_choice, axis_output]))

## Exercise 1 — read the real shapes / Ejercicio 1 — lee las formas reales

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

For each real tensor, three steps.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">FOR EVERY TENSOR · PARA CADA TENSOR</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Print its shape and its order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Name every axis.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Decide whether reordering axis 0 would preserve or change the meaning.</div></div>

One question settles the third step:

**Is axis 0 a collection of independent examples, or part of the internal
structure of one observation?**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Para cada tensor: <b>1 ·</b> imprime forma y orden, <b>2 ·</b> nombra cada eje, <b>3 ·</b> decide si reorganizar el eje 0 conserva o cambia el significado.<br><br>La pregunta que lo resuelve: ¿el eje 0 son ejemplos independientes, o parte de la estructura interna de una sola observación?</div>

In [ ]:
# TODO 1 / TAREA 1
#
# Inspect / Inspecciona:
#   real_digit
#   digit_batch
#   real_photo
#   real_video
#
# EN:
# 1. Print .shape and .ndim.
# 2. Name every axis.
# 3. Explain what would happen if axis 0 were reordered.
#
# ES:
# 1. Imprime .shape y .ndim.
# 2. Nombra cada eje.
# 3. Explica qué ocurriría si se reorganizara el eje 0.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

objects = [
    (
        "one real digit / un dígito real",
        real_digit,
        "(H, W)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real digit batch / lote real de dígitos",
        digit_batch,
        "(N, H, W)",
        "axis 0 is independent examples; batch order can change",
        "el eje 0 son ejemplos independientes; el orden del lote puede cambiar",
    ),
    (
        "real RGB photo / foto RGB real",
        real_photo,
        "(H, W, C)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real sampled video / video real muestreado",
        real_video,
        "(T, H, W, C)",
        "axis 0 is time; reordering it changes chronology",
        "el eje 0 es tiempo; reorganizarlo cambia la cronología",
    ),
]

for name, arr, axes, en, es in objects:
    print(name)
    print("  shape/forma:", arr.shape)
    print("  order/orden:", arr.ndim)
    print("  axes/ejes:", axes)
    print("  EN:", en)
    print("  ES:", es)
    print()

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

The count of axes gives the order. The dataset gives the axes their meaning.

- **batch axis** — independent observations.
- **spatial axis** — position inside one image.
- **time axis** — position in an ordered sequence.
- **channel axis** — different measurements at the same position.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><b>eje de lote</b> — observaciones independientes.</li><li style="margin:.35em 0"><b>eje espacial</b> — posición dentro de una imagen.</li><li style="margin:.35em 0"><b>eje temporal</b> — posición en una secuencia ordenada.</li><li style="margin:.35em 0"><b>eje de canales</b> — mediciones distintas en la misma posición.</li></ul></div>

</details>

### Compare the two meanings / Compara los dos significados

Switch between **Batch / Lote** and **Time / Tiempo**. Same numbers, different
reading.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cambia entre <b>Lote</b> y <b>Tiempo</b>: los mismos números, otra lectura.</div>

In [ ]:
#@title 🔍 Batch vs time / Lote contra tiempo — run me / ejecútame { display-mode: 'form' }

meaning_toggle = widgets.ToggleButtons(
    options=[
        ("Batch / Lote", "batch"),
        ("Time / Tiempo", "time"),
    ],
    value="batch",
    description="Meaning / Significado:",
    style={"description_width": "140px"},
)

def show_same_shape_meaning(kind):
    plt.close("all")

    if kind == "batch":
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(digit_batch[i], cmap="gray")
            ax.set_title(f"N={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = independent examples / "
            "Misma forma: eje 0 = ejemplos independientes"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering N changes presentation order, not the identity of the examples.")
        print("ES: reorganizar N cambia el orden de presentación, no la identidad de los ejemplos.")

    else:
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(video_patch[i], cmap="gray")
            ax.set_title(f"T={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = ordered time / "
            "Misma forma: eje 0 = tiempo ordenado"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering T changes chronology.")
        print("ES: reorganizar T cambia la cronología.")

meaning_output = widgets.interactive_output(
    show_same_shape_meaning,
    {"kind": meaning_toggle},
)

display(widgets.VBox([meaning_toggle, meaning_output]))

### See the shuffle / Observa la reorganización

Compare four orders: digits before and after, video before and after.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Compara cuatro órdenes: dígitos antes y después, video antes y después.</div>

In [ ]:
#@title 🔀 Shuffle viewer / Visor de reorganización — run me / ejecútame { display-mode: 'form' }

# One shared shuffle to visualize (TODO 2 steps 1-3). The comparison and the
# explanation of *why* it matters stay in the folded solution above.
perm = rng.permutation(len(digit_batch))
shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

shuffle_view = widgets.Dropdown(
    options=[
        ("Digits — original / Dígitos — original", "digits_original"),
        ("Digits — shuffled / Dígitos — reorganizados", "digits_shuffled"),
        ("Video — original / Video — original", "video_original"),
        ("Video — shuffled / Video — reorganizado", "video_shuffled"),
    ],
    value="digits_original",
    description="View / Vista:",
    style={"description_width": "100px"},
)

def show_shuffle_view(view):
    plt.close("all")
    fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))

    if view == "digits_original":
        arr = digit_batch
        labels = digit_labels
        title = "Independent examples — original order / Ejemplos independientes — orden original"
        cmap = "gray"
    elif view == "digits_shuffled":
        arr = shuffled_digits
        labels = shuffled_labels
        title = "Independent examples — shuffled order / Ejemplos independientes — orden reorganizado"
        cmap = "gray"
    elif view == "video_original":
        arr = video_patch
        labels = np.arange(8)
        title = "Time sequence — original order / Secuencia temporal — orden original"
        cmap = "gray"
    else:
        arr = shuffled_video
        labels = perm
        title = "Time sequence — shuffled order / Secuencia temporal — orden reorganizado"
        cmap = "gray"

    for i, ax in enumerate(axes):
        ax.imshow(arr[i], cmap=cmap)
        ax.set_title(str(labels[i]))
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

shuffle_output = widgets.interactive_output(
    show_shuffle_view,
    {"view": shuffle_view},
)

display(widgets.VBox([shuffle_view, shuffle_output]))

## 2.3 Real videos have different lengths / Los videos reales tienen longitudes distintas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

A batch is one rectangular block. Real sequences are not: clip A has 4 frames,
clip B has 7, clip C has 5.

$$
\text{padded length } L = \max_i T_i
\qquad
\text{wasted cells} = \sum_i \bigl(L - T_i\bigr) \cdot H \cdot W \cdot C
$$

Read it as: every clip is stretched to the longest one, and everything you
added is a cell the model will read unless a mask tells it not to. For 4, 7
and 5 frames, $L = 7$, so the batch holds $3 \times 7 = 21$ frame slots: 16 real frames and 5 of padding.


**Padding** is the usual answer.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">PADDING, IN FOUR MOVES · PADDING EN CUATRO PASOS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Take the longest length — <code>7</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Copy each real clip into a 7-frame slot.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Fill the unused positions with a placeholder.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span>Store a <b>mask</b> saying which positions are real.</div></div>

Three students answer 4, 7 and 5 questions. The spreadsheet demands 7 columns
from everyone. The blank cells are **not answers** — and something has to say
so. That something is the mask.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un lote es un bloque rectangular; las secuencias reales no lo son. El padding crea las posiciones que faltan y la máscara dice cuáles contienen datos reales. Tres estudiantes responden 4, 7 y 5 preguntas: si la hoja exige 7 columnas para todos, las celdas vacías <b>no son respuestas</b>.</div>

## Exercise 3 — build an order-5 batch / Ejercicio 3 — construye un lote de orden 5

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Three non-overlapping segments of the real video: 4 frames, 7 frames, 5 frames.

Predict all five answers before you pad.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>The final shape.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Which axis is <code>N</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Which axis is <code>T</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span>How many <code>(N, T)</code> positions are real.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span>How many are padding.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tres segmentos del video real — 4, 7 y 5 fotogramas. Antes de ejecutar, predice: la forma final, qué eje es <code>N</code>, qué eje es <code>T</code>, cuántas posiciones <code>(N, T)</code> son reales y cuántas son padding.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Spatially subsample real_video with real_video[:, ::4, ::4, :].
# 2. Build three non-overlapping clips with lengths 4, 7, and 5.
# 3. Compute T_max.
# 4. Allocate padded with shape (N, T_max, H, W, C).
# 5. Build a Boolean validity mask with shape (N, T_max).
# 6. Count measured slots and padding slots.
#
# ES:
# 1. Submuestrea espacialmente real_video con real_video[:, ::4, ::4, :].
# 2. Construye tres clips no superpuestos de longitudes 4, 7 y 5.
# 3. Calcula T_max.
# 4. Crea padded con forma (N, T_max, H, W, C).
# 5. Construye una máscara booleana de validez con forma (N, T_max).
# 6. Cuenta posiciones medidas y posiciones de padding.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

video_small = real_video[:, ::4, ::4, :]  # measured pixels, spatially subsampled

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

padded_slots = int((~valid).sum())
total_slots = int(valid.size)
measured_slots = int(valid.sum())

print("Clip lengths / Longitudes:", lengths.tolist())
print("Padded shape / Forma con padding:", padded.shape)
print("Order / Orden:", padded.ndim)
print("Axes / Ejes: (N, T, H, W, C)")
print("Validity mask / Máscara de validez:", valid.shape)
print("Measured frame slots / Posiciones medidas:", measured_slots)
print("Padding frame slots / Posiciones de padding:", padded_slots)
print(f"Padding fraction / Fracción de padding: {padded_slots / total_slots:.1%}")

assert padded.shape == (3, 7, 135, 240, 3)
assert valid.sum() == 16

In [ ]:
# The notebook builds the padded order-5 batch here, in a visible cell, so the
# REAL/PAD map and the padding explorer below run whether or not the folded
# solution was executed. The slot-count analysis and the asserts stay folded.
video_small = real_video[:, ::4, ::4, :]

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

### REAL vs PAD map / Mapa REAL vs PAD

The map shows the batch at the `(N, T)` level.

- **REAL** — a measured frame exists here.
- **PAD** — none does; the position only keeps the batch rectangular.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El mapa muestra el lote en el nivel <code>(N, T)</code>. <b>REAL</b> = existe un fotograma medido. <b>PAD</b> = no existe; esa posición solo mantiene rectangular el lote.</div>

In [ ]:
#@title 🗺️ REAL vs PAD map / Mapa REAL contra PAD — run me / ejecútame { display-mode: 'form' }

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.imshow(valid, cmap="Greys", vmin=0, vmax=1, aspect="auto")

for n in range(N):
    for t in range(T_max):
        text = "REAL" if valid[n, t] else "PAD"
        text_color = "white" if valid[n, t] else "black"
        ax.text(
            t,
            n,
            text,
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
            fontweight="bold",
        )

ax.set_xticks(range(T_max))
ax.set_xlabel("Time slot T / Posición temporal T")
ax.set_yticks(range(N))
ax.set_yticklabels(
    [f"clip {n} · measured T={lengths[n]}" for n in range(N)]
)
ax.set_ylabel("Clip N / Video N")
ax.set_title(
    "Measured frames vs padding / Fotogramas medidos vs padding"
)
plt.tight_layout()
plt.show()

### Padding explorer / Explorador de padding

Pick a clip and a time position. A real slot shows its measured frame.

A padded slot shows a **PAD card**, not a black image — because a black image
is exactly what a real measurement of darkness would look like.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">TRY BOTH · PRUEBA LAS DOS</div><div style="margin:.55em 0"><code>Clip N = 0, Time T = 0</code> → REAL</div><div style="margin:.55em 0"><code>Clip N = 2, Time T = 6</code> → PAD</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un clip y una posición. Si es real verás el fotograma medido; si es padding verás una tarjeta <b>PAD</b>, no una imagen negra — una imagen negra es justo lo que parecería una medición real de oscuridad.</div>

In [ ]:
#@title 🔍 Padding explorer / Explorador de padding — run me / ejecútame { display-mode: 'form' }

clip_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=N - 1,
    step=1,
    description="Clip N / Video N:",
    continuous_update=False,
    style={"description_width": "120px"},
)

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=T_max - 1,
    step=1,
    description="Time T / Tiempo T:",
    continuous_update=False,
    style={"description_width": "120px"},
)

def explore_padding(clip_idx, time_idx):
    slot = padded[clip_idx, time_idx]
    is_valid = bool(valid[clip_idx, time_idx])

    print(
        f"Index / Índice: padded[{clip_idx}, {time_idx}] | "
        f"shape/forma={slot.shape} | valid/válido={is_valid}"
    )

    fig, ax = plt.subplots(figsize=(6.4, 3.6))

    if is_valid:
        ax.imshow(slot)
        ax.set_title("REAL frame / Fotograma REAL")
        ax.axis("off")
        print("EN: a measured frame exists at this position.")
        print("ES: existe un fotograma medido en esta posición.")
    else:
        ax.set_facecolor("#f2f2f2")
        ax.text(
            0.5,
            0.58,
            "PAD",
            ha="center",
            va="center",
            fontsize=34,
            fontweight="bold",
            color="#993333",
            transform=ax.transAxes,
        )
        ax.text(
            0.5,
            0.38,
            "No measured frame exists here\n"
            "No existe un fotograma medido aquí",
            ha="center",
            va="center",
            fontsize=11,
            transform=ax.transAxes,
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(
            "Padding only keeps the batch rectangular / "
            "El padding solo mantiene el lote rectangular"
        )
        print("EN: this slot is padding, not a measured black frame.")
        print("ES: esta posición es padding, no un fotograma negro medido.")

    plt.tight_layout()
    plt.show()

padding_output = widgets.interactive_output(
    explore_padding,
    {
        "clip_idx": clip_slider,
        "time_idx": time_slider,
    },
)

display(
    widgets.VBox([
        widgets.HBox([clip_slider, time_slider]),
        padding_output,
    ])
)

<details>
<summary><strong>Why padding needs a mask / Por qué el padding necesita una máscara</strong></summary>

The padded tensor has a convenient rectangular shape. Not every `(N, T)`
position holds a measured frame.

The mask answers one question: **should the model treat this position as real
data?**

Without it, the zeros of padding are indistinguishable from genuine
observations.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El tensor con padding es rectangular y cómodo, pero no toda posición <code>(N, T)</code> contiene un fotograma medido. La máscara responde una pregunta: <b>¿debe el modelo tratar esta posición como dato real?</b> Sin ella, los ceros del relleno son indistinguibles de observaciones reales.</div>

</details>

## 2.4 The same idea in science / La misma idea en ciencia

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Nothing here is about video. Picture a microscope recording cells over time —
the tensor could be <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N, T, H, W, C)</span>:

- `N` — patient, dish, well or field of view;
- `T` — measurement time;
- `H`, `W` — image height and width;
- `C` — imaging channels.

The convention depends on the experiment, so it has to be written down.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">A TRAP · UNA TRAMPA</div>A <b>field of view</b> is not automatically <code>H</code> or <code>W</code>. Those two are pixel coordinates <i>inside one image</i>.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La misma lógica vale en ciencia: <code>N</code> puede ser paciente, plato, pozo o campo de visión; <code>T</code> el tiempo de medición; <code>H</code> y <code>W</code> el alto y el ancho; <code>C</code> los canales. La convención depende del experimento y debe documentarse. Un <b>campo de visión</b> no es automáticamente <code>H</code> ni <code>W</code>: esos son coordenadas de píxel dentro de una imagen.</div>

### Experimental-axis explorer / Explorador de ejes experimentales

Pick an axis. Read what it could mean in a microscopy experiment.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un eje y observa qué podría significar en un experimento de microscopía.</div>

In [ ]:
#@title 🔬 Experimental-axis explorer / Explorador de ejes experimentales — run me / ejecútame { display-mode: 'form' }

experiment_axis = widgets.ToggleButtons(
    options=["N", "T", "H", "W", "C"],
    value="N",
    description="Axis / Eje:",
    style={"description_width": "90px"},
)

def explain_experiment_axis(axis):
    explanations = {
        "N": (
            "independent observation: patient, well, dish, or field of view",
            "observación independiente: paciente, pozo, plato o campo de visión",
        ),
        "T": (
            "measurement time or acquisition step",
            "tiempo de medición o paso de adquisición",
        ),
        "H": (
            "pixel rows inside one image",
            "filas de píxeles dentro de una imagen",
        ),
        "W": (
            "pixel columns inside one image",
            "columnas de píxeles dentro de una imagen",
        ),
        "C": (
            "colour, stain, fluorescence, or other measurement channels",
            "canales de color, tinción, fluorescencia u otras mediciones",
        ),
    }

    en, es = explanations[axis]
    print(f"{axis}")
    print("EN:", en)
    print("ES:", es)

experiment_output = widgets.interactive_output(
    explain_experiment_axis,
    {"axis": experiment_axis},
)

display(widgets.VBox([experiment_axis, experiment_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

One rule, learned on real image and video data: **every axis needs a meaning**.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">FOUR IDEAS · CUATRO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Same shape, different things.</b> <code>(N,H,W)</code> and <code>(T,H,W)</code> can be numerically identical.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>Batch and time are not interchangeable.</b> Reordering examples is not reordering chronology.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>Padding is not measured data.</b> A validity mask says which positions are real.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Higher order just means more axes.</b> Not a better model, not a harder one.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una regla, aprendida sobre datos reales: <b>cada eje necesita un significado</b>. <b>1 ·</b> la misma forma puede significar cosas distintas; <b>2 ·</b> lote y tiempo no son intercambiables; <b>3 ·</b> el relleno no son datos medidos; <b>4 ·</b> más orden es solo más ejes.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **03 · Indexing and broadcasting real data / Indexación y broadcasting con datos reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)